# 3_gather_data_to_forecast

In [1]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split, TimeSeriesSplit


import os
import sys
from dotenv import load_dotenv
#import json
import pandas as pd
import numpy as np
from google.cloud import bigquery
import pydata_google_auth
#import requests
#import time
from datetime import date#, timedelta, datetime
import urllib3
from pathlib import Path

load_dotenv()
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# adding project directory for easier import to notebook
notebook_dir = Path(os.getcwd())
parent_dir = str(notebook_dir.parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from scripts.data_pull_functions import pull_yes_forecast_historical, pull_yes_actual_historical, pull_unit_availability
from scripts.data_clean_functions import clean_yes_forecast

In [2]:
historic_data = pd.read_csv('../data/processed-data/merged_df.csv', )
historic_data['datetime'] = pd.to_datetime(historic_data['datetime'])
historic_data['gas_day'] = pd.to_datetime(historic_data['gas_day'])
historic_data['hourly_gas_burn_MMBtu'] = historic_data["hourly_gas_burn_MMBtu"].fillna(0)

historic_data['hourly_gas_burn_MMBtu'] =  (historic_data['daily_gas_burn_MMBtu'] / historic_data['daily_site_gen_mw']) * historic_data['hourly_site_gen_mw']
historic_data['hourly_gas_burn_MMBtu'] = historic_data['hourly_gas_burn_MMBtu'].fillna(0)

#adding time features
historic_data["hour"] = historic_data["datetime"].dt.hour
historic_data["day_of_week"] = historic_data["datetime"].dt.dayofweek
historic_data["day"] = historic_data["datetime"].dt.day
historic_data["month"] = historic_data["datetime"].dt.month
historic_data['year'] = historic_data['datetime'].dt.year


#adding time features
historic_data["gas_hour"] = historic_data["gas_day"].dt.hour
historic_data["gas_day_of_week"] = historic_data["gas_day"].dt.dayofweek
historic_data["gas_month"] = historic_data["gas_day"].dt.month
historic_data['gas_year'] = historic_data['gas_day'].dt.year

historic_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 128634 entries, 0 to 128633
Data columns (total 31 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   datetime                128634 non-null  datetime64[us]
 1   gas_day                 128634 non-null  datetime64[us]
 2   hour                    128634 non-null  int32         
 3   site                    128634 non-null  str           
 4   hourly_site_gen_mw      128634 non-null  float64       
 5   daily_gas_burn_MMBtu    128634 non-null  float64       
 6   daily_site_gen_mw       128634 non-null  float64       
 7   availability_mw         90855 non-null   float64       
 8   load_forecast           128614 non-null  float64       
 9   net_load_forecast       128614 non-null  float64       
 10  wind_forecast           128614 non-null  float64       
 11  temperature_forecast    128614 non-null  float64       
 12  wind_speed_forecast     128614 non-null  

In [3]:
max_historic_date = historic_data['datetime'].max() - pd.Timedelta(365, unit='D')
max_historic_date

Timestamp('2025-08-01 00:00:00')

In [186]:
training_df = historic_data[historic_data['datetime'] <= max_historic_date]
holdout_df = historic_data[historic_data['datetime'] > max_historic_date]

In [187]:
features = ['datetime', 'gas_day', 'year', 'month', 'day_of_week', 'day', 'hour', 'site', 'wind_speed_actual', 'temperature_actual']

X_training_df = training_df.loc[:,features]
y_training_df = training_df.loc[:,['site', 'hourly_gas_burn_MMBtu']]

In [188]:
models = {}
results = {}
site_hourly_predictions ={}

sites = historic_data["site"].unique()

In [217]:
for site in sites:

    print(f"\nTraining model for: {site}")

    X_train_site = X_training_df[X_training_df["site"] == site].drop(['site', 'datetime', 'gas_day'], axis=1)
    y_train_site = y_training_df[y_training_df['site']== site]['hourly_gas_burn_MMBtu']

    #X_test_site = X_test[X_test["site"] == site].drop(['site', 'datetime', 'gas_day'], axis=1)
    #y_test_site = y_test[y_test['site'] == site]['hourly_gas_burn']
    # CALEB NOTE: Rethink how we split this
    # split_idx = int(len(site_df) * 0.8)

    #train = site_df.iloc[:split_idx]
    #test  = site_df.iloc[split_idx:]
    #print(X_test.head())

    # print(site, "X_train:", X_train_site.shape, "y_train:", y_train_site.shape)
    # print(site, "X_test:", X_test_site.shape, "y_train:", y_test_site.shape)
    
    
    # CALEB NOTE: Should we do some type of tuning??
    model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)

    model.fit(X_train_site, y_train_site)

    #preds = model.predict(X_train_site)
    #mae = mean_absolute_error(y_train_site, preds)

    #print(f"{site} MAE:", mae)

    models[site] = model
    #results[site] = mae
    '''
    sites_df = pd.DataFrame.from_dict({
            'datetime': X_test[X_test["site"] == site]['datetime'],
            'HE': "HE" + (X_test[X_test["site"] == site]['datetime'].dt.hour + 1).map(lambda x:f"{x:02d}"),
            'gas_day': X_test[X_test["site"] == site]['gas_day'],
            'site': site,
            'predicted_gas_burn': preds,
            'actual_gas_burn': y_test_site.values
        })
    '''

    #site_hourly_predictions[site] = sites_df


Training model for: PGS

Training model for: LCS

Training model for: GGS

Training model for: DCS

Training model for: CGS


In [280]:
models['GGS']

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


## Constructing the future dataframe



This section is to create a dataframe consisting of future dates that are to be forecasted. Potential data sources include:
- Yes Energy Forecasted values
- Availability Data from Clarissa's spreadsheet

Steps:
1. pull the future site availability
2. pull the yes energy forecasts
3. merge future site availability and yes energy data together
4. construct date time attributes that can be used by the model

In [53]:
start_dt = '2026-08-08'
end_dt = '2026-08-14'

In [ ]:
def pull_data_to_forecast(forecast_start, forecast_end, availability_file: str = "", forecast_features: list = []) -> pd.DataFrame:
    """
    Function pulls data based on a date range that is intended to be forecasted.

    Parameters:
        forecast_start: first date to be included in the data
        forecast_end: last date to be included in the data
        availability_file: file location containing hourly unit level MW availability. If left blank, function will default to the most recently created file in the Transposed Forward Looking Data folder
        forecast_features: list of column names that should be returned. Will likely need to align with the 'feature_names_in_' attribute from the predictive model

    Returns:
        dataframe containing data to be forecasted
    """

    if availability_file:
        latest_file = availability_file
    else:
        directory = Path('G:/Trading/Forecasts/Daily Gas Burn Forecast by Site/Transposed Forward Looking Data')
        latest_file = str(max(directory.glob('*'), key=lambda f: f.stat().st_birthtime))

    future_site_availability_df = pull_unit_availability(excel_files=[latest_file], csv_files=[], sheet='Transposed', start_date=forecast_start, end_date=forecast_end)['site_availability_df']

    future_yes_data_df = pull_yes_forecast_historical(os.getenv('YES_USERNAME'), os.getenv('YES_PASSWORD'), forecast_start, forecast_end)
    future_yes_data_df = clean_yes_forecast(future_yes_data_df)

    future_data_to_forecast_df = future_site_availability_df.merge(future_yes_data_df, how='left', on=['datetime'])

    # adding datetime features
    future_data_to_forecast_df["hour"] = future_data_to_forecast_df["datetime"].dt.hour
    future_data_to_forecast_df["day_of_week"] = future_data_to_forecast_df["datetime"].dt.dayofweek
    future_data_to_forecast_df["day"] = future_data_to_forecast_df["datetime"].dt.day
    future_data_to_forecast_df["month"] = future_data_to_forecast_df["datetime"].dt.month
    future_data_to_forecast_df['year'] = future_data_to_forecast_df['datetime'].dt.year
    future_data_to_forecast_df['gas_day'] = (pd.to_datetime(future_data_to_forecast_df["datetime"]) - pd.Timedelta(hours=10)).dt.date
    future_data_to_forecast_df['hour_end'] = future_data_to_forecast_df['hour'] + 1
    future_data_to_forecast_df['hour_end'] = 'HE' + future_data_to_forecast_df['hour_end'].astype(str)

    future_data_to_forecast_df = future_data_to_forecast_df.loc[:, forecast_features] if forecast_features else future_data_to_forecast_df

    return future_data_to_forecast_df

In [60]:
function_check = pull_future_data_to_forecast(start_dt, end_dt, ['datetime', 'month'])

Pulling forecast: 2026-08-08 ---> 2026-08-14


In [61]:
function_check.shape

(600, 2)

In [15]:
future_features = ['year', 'month', 'day_of_week', 'day', 'hour', 'wind_speed_forecast', 'temperature_forecast']
future_data = future_data.loc[:, future_features]
future_data = future_data.rename(columns = {'wind_speed_forecast': 'wind_speed_actual', 'temperature_forecast': 'temperature_actual'})
future_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 167 entries, 121 to 287
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   year                167 non-null    int32  
 1   month               167 non-null    int32  
 2   day_of_week         167 non-null    int32  
 3   day                 167 non-null    int32  
 4   hour                167 non-null    int32  
 5   wind_speed_actual   167 non-null    float64
 6   temperature_actual  167 non-null    float64
dtypes: float64(2), int32(5)
memory usage: 6.0 KB


In [284]:
preds = models['GGS'].predict(future_data)


In [285]:
from datetime import datetime
future_data['preds'] =  preds
future_data['datetime'] = pd.to_datetime(future_data[['year', 'month', 'day', 'hour']])
future_data['gas_day'] = (pd.to_datetime(future_data["datetime"]) - pd.Timedelta(hours=10)).dt.date
future_data.head(12)


,year,month,day_of_week,day,hour,wind_speed_actual,temperature_actual,preds,datetime,gas_day
121,2026,8,3,6,1,8.70,62.960000,120.924591,2026-08-06 01:00:00,2026-08-05
122,2026,8,3,6,2,8.10,61.160000,123.321640,2026-08-06 02:00:00,2026-08-05
123,2026,8,3,6,3,7.45,59.660000,122.233818,2026-08-06 03:00:00,2026-08-05
124,2026,8,3,6,4,11.50,62.043333,128.104355,2026-08-06 04:00:00,2026-08-05
125,2026,8,3,6,5,9.95,60.053333,140.087906,2026-08-06 05:00:00,2026-08-05
126,2026,8,3,6,6,6.80,59.813333,172.026230,2026-08-06 06:00:00,2026-08-05
127,2026,8,3,6,7,6.85,59.096667,206.319717,2026-08-06 07:00:00,2026-08-05
128,2026,8,3,6,8,7.45,63.180000,302.236542,2026-08-06 08:00:00,2026-08-05
129,2026,8,3,6,9,7.75,67.200000,510.491547,2026-08-06 09:00:00,2026-08-05
130,2026,8,3,6,10,8.70,72.620000,700.445557,2026-08-06 10:00:00,2026-08-06


In [286]:
daily_preds = future_data.groupby(by=['gas_day'])['preds'].sum().reset_index()
daily_preds

,gas_day,preds
0,2026-08-05,1825.746338
1,2026-08-06,12980.356445
2,2026-08-07,11598.793945
3,2026-08-08,10116.519531
4,2026-08-09,9037.642578
5,2026-08-10,8888.911133
6,2026-08-11,10921.708984
7,2026-08-12,9037.478516


In [287]:
daily_preds[pd.to_datetime(daily_preds['gas_day']) >= '2026-08-07']

,gas_day,preds
2,2026-08-07,11598.793945
3,2026-08-08,10116.519531
4,2026-08-09,9037.642578
5,2026-08-10,8888.911133
6,2026-08-11,10921.708984
7,2026-08-12,9037.478516


In [201]:
future_data

,year,month,day_of_week,day,hour,wind_speed_actual,temperature_actual,preds,datetime,gas_day
121,2026,8,3,6,1,8.70,62.960000,1037.758667,2026-08-06 01:00:00,2026-08-05
122,2026,8,3,6,2,8.10,61.160000,967.964661,2026-08-06 02:00:00,2026-08-05
123,2026,8,3,6,3,7.45,59.660000,1063.849243,2026-08-06 03:00:00,2026-08-05
124,2026,8,3,6,4,11.50,62.043333,796.993896,2026-08-06 04:00:00,2026-08-05
125,2026,8,3,6,5,9.95,60.053333,831.979553,2026-08-06 05:00:00,2026-08-05
...,...,...,...,...,...,...,...,...,...,...
283,2026,8,2,12,19,10.55,76.220000,1843.991577,2026-08-12 19:00:00,2026-08-12
284,2026,8,2,12,20,9.65,74.060000,1836.780518,2026-08-12 20:00:00,2026-08-12
285,2026,8,2,12,21,7.75,70.580000,1816.382202,2026-08-12 21:00:00,2026-08-12
286,2026,8,2,12,22,7.10,67.160000,1511.676025,2026-08-12 22:00:00,2026-08-12


In [210]:
future_data[pd.to_datetime(future_data['gas_day']) == '2026-08-07']

,year,month,day_of_week,day,hour,wind_speed_actual,temperature_actual,preds,datetime,gas_day
154,2026,8,4,7,10,10.55,63.980000,1543.743408,2026-08-07 10:00:00,2026-08-07
155,2026,8,4,7,11,11.15,67.100000,1522.227783,2026-08-07 11:00:00,2026-08-07
156,2026,8,4,7,12,11.45,70.280000,1819.105469,2026-08-07 12:00:00,2026-08-07
157,2026,8,4,7,13,11.80,73.280000,1847.933472,2026-08-07 13:00:00,2026-08-07
158,2026,8,4,7,14,12.10,75.440000,1785.501709,2026-08-07 14:00:00,2026-08-07
159,2026,8,4,7,15,12.15,77.420000,1790.985352,2026-08-07 15:00:00,2026-08-07
160,2026,8,4,7,16,12.45,78.420000,1665.900146,2026-08-07 16:00:00,2026-08-07
161,2026,8,4,7,17,12.10,79.556667,1777.348145,2026-08-07 17:00:00,2026-08-07
162,2026,8,4,7,18,12.10,78.600000,1747.463745,2026-08-07 18:00:00,2026-08-07
163,2026,8,4,7,19,11.20,78.060000,1804.912964,2026-08-07 19:00:00,2026-08-07
